# 🧩 Notebook 3 · Restaurant — Polymorphism & Patterns

Notebook 2 gave us a working POS. Now we make it *extensible* using three
design patterns you'll meet in every OOD interview:

1. **Strategy** — swap pricing/discount rules (standard, happy-hour, loyalty)
   without touching `Order` or `Bill`.
2. **Factory** — build `Menu` and `MenuItem` objects from configuration
   (JSON / a dict) so non-programmers can edit the menu.
3. **Observer** — when an order is *placed* or *served*, notify the kitchen
   screen, an SMS to the guest, and the manager's dashboard — all decoupled.

> **Why patterns?** They're *names* for shapes you'd invent anyway. Knowing
> the name helps two developers agree on an idea in one word.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/restaurant
uv sync
```

Select the `.venv` kernel. If missing → `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ Shared domain (copied from Notebook 2 so this runs alone)


In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import time
from enum import Enum
from itertools import count


@dataclass(frozen=True)
class MenuItem:
    name: str
    price: float
    category: str
    vegetarian: bool = False


class OrderStatus(Enum):
    OPEN = "open"; PLACED = "placed"; SERVED = "served"; PAID = "paid"


class TableState(Enum):
    FREE = "free"; TAKEN = "taken"; RESERVED = "reserved"


@dataclass
class Table:
    number: int
    seats: int
    state: TableState = TableState.FREE
    def free(self): self.state = TableState.FREE


@dataclass
class OrderItem:
    item: MenuItem
    qty: int = 1
    def line_total(self) -> float: return round(self.item.price * self.qty, 2)


_oids = count(1)

@dataclass
class Order:
    table: Table
    id: int = field(default_factory=lambda: next(_oids))
    items: list[OrderItem] = field(default_factory=list)
    status: OrderStatus = OrderStatus.OPEN
    def add(self, item, qty=1): self.items.append(OrderItem(item, qty))
    def subtotal(self) -> float: return round(sum(i.line_total() for i in self.items), 2)


## 2️⃣ Strategy — swap pricing rules at runtime

### ❌ Bad: hard-coded tax + tip in `Order.pay()`

```python
def pay(self):
    sub = self.subtotal()
    return sub + sub*0.08 + sub*0.15  # magic numbers, one formula forever
```

If marketing runs a "20% off pizzas on Tuesdays" promo, you're editing
`Order.pay()` again. Every promo = another `if`. Ten promos later the method
is unreadable.

### ✅ Best: a `PricingStrategy` interface

The `Order` depends on an **abstract** strategy. You can plug in any rule
without changing the order code. (Open/Closed principle.)


In [ ]:
class PricingStrategy(ABC):
    """Given an order, return the final bill as a dict.

    Contract: the returned dict always has `subtotal`, `tax`, `tip`, `total`.
    Decorating strategies may *add* keys, but must never remove these four.
    """

    @abstractmethod
    def bill(self, order: Order) -> dict: ...


class StandardPricing(PricingStrategy):
    """Subtotal + 8% tax + 15% tip. The one strategy that does the real math."""

    def __init__(self, tax_rate=0.08, tip_rate=0.15):
        self.tax_rate, self.tip_rate = tax_rate, tip_rate

    def bill(self, order: Order) -> dict:
        sub = order.subtotal()
        tax = round(sub * self.tax_rate, 2)
        tip = round(sub * self.tip_rate, 2)
        return {"subtotal": sub, "tax": tax, "tip": tip,
                "total": round(sub + tax + tip, 2)}


# ---------------------------------------------------------------------------
# 🧠 The subtle part: WHERE does a discount apply?
#
# Naive decorators subtract from the *total* the base strategy returned. That is
# wrong in the real world: a discount reduces the taxable amount, so tax and tip
# must be computed on the reduced subtotal, not the full one.
#
# The fix is to change the *input* the base strategy sees, not its output. This
# tiny read-only view is all it takes -- and it means discounts stack correctly.
# ---------------------------------------------------------------------------
class DiscountedOrder:
    """A read-only view of an order whose subtotal is reduced by `amount`."""

    def __init__(self, inner, amount: float):
        self._inner, self._amount = inner, amount

    @property
    def items(self):  return self._inner.items
    @property
    def id(self):     return self._inner.id
    @property
    def table(self):  return self._inner.table

    def subtotal(self) -> float:
        return round(max(0.0, self._inner.subtotal() - self._amount), 2)


class HappyHourPricing(PricingStrategy):
    """20% off drinks between 17:00 and 19:00, applied BEFORE tax and tip."""

    def __init__(self, now: time, base: PricingStrategy):
        self.now, self.base = now, base

    def bill(self, order: Order) -> dict:
        discount = 0.0
        if time(17, 0) <= self.now < time(19, 0):
            discount = round(sum(i.line_total() for i in order.items
                                 if i.item.category == "drinks") * 0.20, 2)
        result = self.base.bill(DiscountedOrder(order, discount))
        result["discount"] = discount
        return result


class LoyaltyPricing(PricingStrategy):
    """Loyalty members get 10% off the subtotal, BEFORE tax and tip.

    This class is a *decorator* over another strategy: it reduces the amount the
    base strategy prices, without caring how the base computes tax or tip. That
    is what lets it wrap `StandardPricing`, `HappyHourPricing`, or another
    discount -- in any order.
    """

    def __init__(self, base: PricingStrategy, member: bool):
        self.base, self.member = base, member

    def bill(self, order: Order) -> dict:
        if not self.member:
            return self.base.bill(order)
        saved = round(order.subtotal() * 0.10, 2)
        result = self.base.bill(DiscountedOrder(order, saved))
        result["loyalty_discount"] = saved
        return result


# --- demo -------------------------------------------------------------------
menu_items = {
    "Margherita": MenuItem("Margherita", 10, "pizza", True),
    "Coke":       MenuItem("Coke",        3, "drinks", True),
}
t = Table(1, 2, TableState.TAKEN)
o = Order(table=t)
o.add(menu_items["Margherita"])
o.add(menu_items["Coke"], qty=2)          # subtotal = 10 + 6 = $16

standard  = StandardPricing()
happy     = HappyHourPricing(now=time(18, 0), base=standard)
loyal     = LoyaltyPricing(base=standard, member=True)
both      = LoyaltyPricing(base=happy, member=True)   # stack them — same interface

print("standard   :", standard.bill(o))
print("happy hour :", happy.bill(o))
print("loyalty    :", loyal.bill(o))
print("both       :", both.bill(o))

### Read the output carefully

| plan | subtotal | tax | total |
|---|---|---|---|
| standard | 16.00 | 1.28 | 19.68 |
| happy hour | **14.80** | **1.18** | 18.20 |
| loyalty | **14.40** | **1.15** | 17.71 |
| both stacked | **13.20** | **1.06** | 16.24 |

The **subtotal and the tax move together**. That is the whole reason `DiscountedOrder`
exists: a decorator that just subtracted from `total` would have left tax at `1.28`
in every row — i.e. the guest would keep paying tax on food they were never charged for.

Two properties fall out of doing it this way:

- **Any strategy can wrap any other.** `LoyaltyPricing(HappyHourPricing(StandardPricing()))`
  works because each layer speaks the same `bill(order) -> dict` language. That is the
  difference between *Strategy* (pick one algorithm) and *Decorator* (compose several).
- **Only `StandardPricing` knows tax rates.** Adding a service charge, a city surcharge,
  or a different tip rule is a change in exactly one class, and every discount keeps working.

💡 **Real-world parallel:** payment processors (Stripe, PayPal) use the same shape —
your checkout code talks to a `PaymentStrategy`, and you swap providers without
touching the cart.

## 3️⃣ Factory — build menus from config

### ❌ Bad: new items are hard-coded in Python

```python
menu = Menu([
    MenuItem("Margherita", 10, "pizza"),
    MenuItem("Pepperoni", 12, "pizza"),
    ...  # change requires a developer + deploy
])
```

Restaurants change menus weekly. Non-developers (the manager) should be
able to edit a JSON file.

### ✅ Best: a `MenuFactory` that reads config


In [ ]:
import json

SAMPLE_CONFIG = """
[
    {"name": "Margherita", "price": 10.0, "category": "pizza", "vegetarian": true},
    {"name": "Pepperoni",  "price": 12.0, "category": "pizza"},
    {"name": "Caesar",     "price":  8.0, "category": "salad",   "vegetarian": true},
    {"name": "Coke",       "price":  3.0, "category": "drinks",  "vegetarian": true},
    {"name": "Tiramisu",   "price":  6.0, "category": "dessert", "vegetarian": true}
]
"""


class MenuFactory:
    """Build MenuItems from a dict/JSON. One place knows the schema."""

    @staticmethod
    def from_dict(d: dict) -> MenuItem:
        required = {"name", "price", "category"}
        missing = required - d.keys()
        if missing:
            raise ValueError(f"menu item missing fields: {missing}")
        return MenuItem(
            name=d["name"],
            price=float(d["price"]),
            category=d["category"],
            vegetarian=bool(d.get("vegetarian", False)),
        )

    @classmethod
    def from_json(cls, text: str) -> list[MenuItem]:
        return [cls.from_dict(d) for d in json.loads(text)]


items = MenuFactory.from_json(SAMPLE_CONFIG)
for mi in items:
    tag = " 🌱" if mi.vegetarian else ""
    print(f"{mi.name:<12} ${mi.price:5.2f}  [{mi.category}]{tag}")

assert any(i.vegetarian for i in items)


💡 **Real-world parallel:** web frameworks like Django use factories to
turn database rows into model instances, and game engines use them to spawn
enemies from level files.


## 4️⃣ Observer — notify many systems when something happens

When an order is *placed*:

- the **kitchen display** should show it,
- the **guest** should get an SMS ("your order is in!"),
- the **manager dashboard** should update "open orders" counter.

### ❌ Bad: `Order.place()` imports all three and calls them

Tight coupling. Changing the SMS provider changes `Order`. Testing `Order`
requires mocking three modules.

### ✅ Best: `Order` emits events; subscribers register themselves


In [ ]:
class OrderEvents:
    """Tiny publish/subscribe hub keyed by event name."""

    def __init__(self):
        self._subs: dict[str, list] = {}

    def subscribe(self, event: str, fn) -> None:
        self._subs.setdefault(event, []).append(fn)

    def publish(self, event: str, order: Order) -> None:
        for fn in self._subs.get(event, []):
            fn(order)


@dataclass
class ObservableOrder(Order):
    """Same as Order, but broadcasts lifecycle events.

    The hub is an ordinary *field*, injected per order (dependency injection),
    not a class-level global. Two restaurants — or a test and production — can
    then run side by side with different subscribers, and a test never has to
    remember to reset shared state.
    """

    events: OrderEvents = field(default_factory=OrderEvents)

    def place(self):
        if self.status != OrderStatus.OPEN:
            raise ValueError("order already placed")
        if not self.items:
            raise ValueError("empty order")
        self.status = OrderStatus.PLACED
        self.events.publish("placed", self)

    def serve(self):
        if self.status != OrderStatus.PLACED:
            raise ValueError("order not placed")
        self.status = OrderStatus.SERVED
        self.events.publish("served", self)


# --- subscribers ------------------------------------------------------------
def kitchen_display(order: Order) -> None:
    names = ", ".join(f"{oi.qty}x {oi.item.name}" for oi in order.items)
    print(f"[KITCHEN]  order {order.id} → {names}")


def sms_to_guest(order: Order) -> None:
    print(f"[SMS]      order {order.id}: your food is on the way 🍕")


def manager_dashboard(order: Order) -> None:
    print(f"[DASH]     table {order.table.number} event: {order.status.value}")


# --- wiring -----------------------------------------------------------------
hub = OrderEvents()
hub.subscribe("placed", kitchen_display)
hub.subscribe("placed", sms_to_guest)
hub.subscribe("placed", manager_dashboard)
hub.subscribe("served", manager_dashboard)

t2 = Table(2, 4, TableState.TAKEN)
o2 = ObservableOrder(table=t2, events=hub)   # the hub is injected, not global
o2.add(menu_items["Margherita"])
o2.add(menu_items["Coke"])
o2.place()
o2.serve()

# An order with no hub still works — subscribers are optional, never required.
solo = ObservableOrder(table=Table(3, 2, TableState.TAKEN))
solo.add(menu_items["Coke"])
solo.place()
print("[SOLO]     an order with its own empty hub placed fine:", solo.status.value)

💡 **Real-world parallel:** every modern chat/social app works this way.
When you post a message, the backend *publishes* a `message_created` event;
separate services handle notifications, feed updates, analytics, and search
indexing — each one added without touching the others.


## 5️⃣ All three patterns together

In production you'd combine them: the **Factory** builds menus from config,
the **Strategy** calculates the bill, and the **Observer** fans events out to
downstream systems. Each pattern protects you from a different kind of
change.

| Change that happens a lot | Pattern that absorbs it |
|---|---|
| New discount / tax rules | **Strategy** |
| New menu items daily | **Factory** |
| New downstream system (Slack alert, loyalty CRM) | **Observer** |

## 🎓 Interview-style recap

If an interviewer asks *"how would you design a restaurant POS?"* — walk
them through:

1. **Domain model** (Notebook 1): Menu, Table, Order, Bill, Staff.
2. **State machines** (Notebook 2) to rule out invalid transitions.
3. **Patterns** (this notebook) for the parts that change most.

That's a 20-minute answer that shows you think about *evolution*, not just
one-shot code.

## 🔭 Further practice

- Add a **Decorator** to `MenuItem` for "extra cheese / gluten-free" that
  bumps the price.
- Use the **State pattern** to replace the `_require` checks in `Order`
  with one class per status.
- Add a **Repository** (SQLite) that persists orders and bills.


## 6️⃣ 🧪 Verify the patterns

A lab that *names* Strategy, Factory, and Observer owes you proof it implemented them.
Each block below checks the property that makes the pattern worth using at all.

In [ ]:
def must_raise(exc, fn, *a, **kw):
    try:
        fn(*a, **kw)
    except exc:
        return True
    raise AssertionError(f'expected {exc.__name__}, nothing was raised')

# ============ 1. STRATEGY ==================================================
demo = Order(table=Table(9, 2, TableState.TAKEN))
demo.add(menu_items["Margherita"])          # $10 pizza
demo.add(menu_items["Coke"], qty=2)         # $6  drinks   -> subtotal $16
assert demo.subtotal() == 16

# The interface is a real abstraction, not a naming convention.
must_raise(TypeError, PricingStrategy)
class Incomplete(PricingStrategy): pass
must_raise(TypeError, Incomplete)

# Every strategy honours the same contract -> callers need no isinstance checks.
strategies = {
    "standard": StandardPricing(),
    "happy":    HappyHourPricing(time(18, 0), StandardPricing()),
    "loyalty":  LoyaltyPricing(StandardPricing(), member=True),
    "stacked":  LoyaltyPricing(HappyHourPricing(time(18, 0), StandardPricing()), True),
    "closed":   HappyHourPricing(time(21, 0), StandardPricing()),   # outside happy hour
    "nonmember": LoyaltyPricing(StandardPricing(), member=False),
}
for name, s in strategies.items():
    b = s.bill(demo)
    assert {"subtotal", "tax", "tip", "total"} <= b.keys(), f'{name} broke the contract'

# Discounts are PRE-TAX: tax must shrink with the subtotal, not stay fixed.
std   = strategies["standard"].bill(demo)
happy = strategies["happy"].bill(demo)
assert std["tax"] == 1.28 and std["total"] == 19.68, std
assert happy["discount"] == 1.20                      # 20% of the $6 of drinks
assert happy["subtotal"] == 14.80                     # base priced the REDUCED order
assert happy["tax"] < std["tax"], 'a pre-tax discount must reduce the tax too'

# Outside happy hour the same object charges full price — behaviour, not config.
assert strategies["closed"].bill(demo)["total"] == std["total"]
# A non-member gets exactly the base bill back.
assert strategies["nonmember"].bill(demo) == std

# Decorators compose: stacking is strictly cheaper than either one alone.
stacked = strategies["stacked"].bill(demo)
assert stacked["total"] < happy["total"] < std["total"], (stacked, happy, std)
assert "discount" in stacked and "loyalty_discount" in stacked

# ...and no strategy ever mutates the order it prices.
assert demo.subtotal() == 16, 'pricing must be a pure read of the order'

# ============ 2. FACTORY ===================================================
built = MenuFactory.from_json(SAMPLE_CONFIG)
assert len(built) == 5 and all(isinstance(m, MenuItem) for m in built)
# The factory owns the schema, so the defaults live in exactly one place.
assert MenuFactory.from_dict({"name": "X", "price": "9.5", "category": "pizza"}) \
       == MenuItem("X", 9.5, "pizza", False)
assert isinstance(MenuFactory.from_dict({"name": "X", "price": "9.5",
                                         "category": "pizza"}).price, float), 'coerced'
# Bad config fails at the boundary with a useful message, not deep inside the app.
must_raise(ValueError, MenuFactory.from_dict, {"name": "X"})
must_raise(ValueError, MenuFactory.from_dict, {"name": "X", "price": 1})

# ============ 3. OBSERVER ==================================================
seen = []
hub_a, hub_b = OrderEvents(), OrderEvents()
hub_a.subscribe("placed", lambda o: seen.append(("a1", o.id)))
hub_a.subscribe("placed", lambda o: seen.append(("a2", o.id)))   # fan-out
hub_a.subscribe("served", lambda o: seen.append(("a-served", o.id)))
hub_b.subscribe("placed", lambda o: seen.append(("b1", o.id)))

oa = ObservableOrder(table=Table(20, 2, TableState.TAKEN), events=hub_a)
oa.add(menu_items["Coke"]); oa.place(); oa.serve()

# Every subscriber of the right event fired, exactly once, in subscription order.
assert [tag for tag, _ in seen] == ["a1", "a2", "a-served"], seen

# Hubs are isolated: hub_b never heard about hub_a's order. A class-level hub
# would have leaked here — that is why `events` is an injected field.
ob = ObservableOrder(table=Table(21, 2, TableState.TAKEN), events=hub_b)
ob.add(menu_items["Coke"]); ob.place()
assert [tag for tag, _ in seen] == ["a1", "a2", "a-served", "b1"], seen

# The subject knows nothing about its subscribers beyond "callable".
assert "kitchen_display" not in "".join(ObservableOrder.place.__doc__ or "")
# Adding a brand-new consumer requires zero changes to ObservableOrder.
audit = []
hub_a.subscribe("placed", audit.append)
oc = ObservableOrder(table=Table(22, 2, TableState.TAKEN), events=hub_a)
oc.add(menu_items["Coke"]); oc.place()
assert audit == [oc]

# The state machine still guards transitions even with events in the mix.
must_raise(ValueError, oc.place)                                    # already placed
must_raise(ValueError, ObservableOrder(table=Table(23, 2)).place)   # empty order

print("Strategy ✅  Factory ✅  Observer ✅  — all three verified")